# Coral Bleaching Map
Data from Harvard Coral Bleaching database [1]

In [1]:
import os
import shutil

# Check if directory is set up properly: If not clone repo
# Check for CoralBleaching.xlsx
# Check if the directory exists
if not os.path.exists("./CoralBleaching.xlsx"):
    # Clone the repository if the file is missing
    if not os.path.exists("./coral-bleaching-repo"):
        !git clone https://github.com/danielstebbings/EE581-Project
    # Move the file to the current directory
    shutil.move("./coral-bleaching-repo/CoralBleaching.xlsx", "./CoralBleaching.xlsx")

In [2]:
# Read Database excel
import polars as pl
bleach_db = pl.read_excel("./CoralBleaching.xlsx")

print(bleach_db.columns)

print(bleach_db["COUNTRY","SEVERITY_CODE"].drop_nulls().sort("SEVERITY_CODE",descending=True))

['ID', 'REGION', 'SUBREGION', 'COUNTRY', 'LOCATION', 'LAT', 'LON', 'MONTH', 'YEAR', 'DEPTH', 'SEVERITY_CODE', 'BLEACHING_SEVERITY', 'CORAL_FAMILY', 'CORAL_SPECIES', 'PERCENTAGE_AFFECTED', 'BLEACHING_DURATION', 'MORTALITY_CODE', 'MORTALITY', 'RECOVERY_CODE', 'RECOVERY', 'SURVEY_TYPE', 'SURVEY_AREA', 'WATER_TEMPERATURE', 'OTHER_FACTORS', 'REMARKS', 'SOURCE', 'REFERENCE_CODE', 'COUNTRY_CODE']
shape: (6_190, 2)
┌────────────────────────┬───────────────┐
│ COUNTRY                ┆ SEVERITY_CODE │
│ ---                    ┆ ---           │
│ str                    ┆ i64           │
╞════════════════════════╪═══════════════╡
│ Mexico (Pacific)       ┆ 3             │
│ United Kingdom         ┆ 3             │
│ United Kingdom         ┆ 3             │
│ United Kingdom         ┆ 3             │
│ United Kingdom         ┆ 3             │
│ …                      ┆ …             │
│ Hawaiian Islands (USA) ┆ -1            │
│ Hawaiian Islands (USA) ┆ -1            │
│ Hawaiian Islands (USA) ┆ -1 

In [3]:
# Import and authenticate
import geemap
import ee
ee.Authenticate()
ee.Initialize(project="coral-bleaching-479416")



In [ ]:
# Lat Long to EE point
bleach_points_sev = []
bleach_points_med = []
bleach_points_low = []

for i,row in enumerate(bleach_db.iter_rows(named=True)):
    match row["SEVERITY_CODE"]:
        case 3:
            bleach_points_sev.append((row["LON"],row["LAT"]))
        case 2:
            print("Match 2")
            bleach_points_med.append((row["LON"],row["LAT"]))
        case 1:
            bleach_points_low.append((row["LON"],row["LAT"]))

bleach_points_sev = ee.List(bleach_points_sev)
bleach_points_med = ee.List(bleach_points_med)
bleach_points_low = ee.List(bleach_points_low)


def coord2point2feature(point):
    return ee.Feature(ee.Geometry.Point(point))

sev_fc = ee.FeatureCollection(bleach_points_sev.map(coord2point2feature))
med_fc = ee.FeatureCollection(bleach_points_med.map(coord2point2feature))
low_fc = ee.FeatureCollection(bleach_points_low.map(coord2point2feature))




0
0
0
0
0
1
0
2
0
3
-1
4
0
5
-1
6
3
7
2
Match 2
8
2
Match 2
9
1
10
1
11
0
12
2
Match 2
13
2
Match 2
14
2
Match 2
15
2
Match 2
16
3
17
3
18
3
19
3
20
3
21
3
22
3
23
1
24
1
25
1
26
1
27
-1
28
-1
29
3
30
1
31
3
32
1
33
3
34
1
35
2
Match 2
36
2
Match 2
37
3
38
1
39
1
40
3
41
3
42
1
43
1
44
3
45
3
46
3
47
-1
48
1
49
1
50
3
51
3
52
2
Match 2
53
0
54
0
55
2
Match 2
56
1
57
3
58
1
59
2
Match 2
60
2
Match 2
61
1
62
3
63
1
64
1
65
-1
66
0
67
3
68
3
69
0
70
3
71
3
72
3
73
3
74
2
Match 2
75
1
76
1
77
1
78
1
79
1
80
1
81
1
82
1
83
1
84
1
85
0
86
-1
87
0
88
1
89
1
90
1
91
2
Match 2
92
3
93
3
94
2
Match 2
95
2
Match 2
96
-1
97
3
98
-1
99
2
Match 2
100
3
101
2
Match 2
102
3
103
1
104
3
105
3
106
3
107
-1
108
2
Match 2
109
0
110
0
111
3
112
2
Match 2
113
2
Match 2
114
2
Match 2
115
2
Match 2
116
2
Match 2
117
1
118
2
Match 2
119
2
Match 2
120
1
121
2
Match 2
122
3
123
3
124
3
125
3
126
2
Match 2
127
2
Match 2
128
2
Match 2
129
2
Match 2
130
3
131
2
Match 2
132
3
133
2
Match 2
134
3
135
3
136
3
137
2
Ma

In [5]:
Map = geemap.Map(center=(0, 0), zoom=2.5)
Map.add_basemap("HYBRID")

Map.add_layer(low_fc.draw(color="FFD900", strokeWidth=5), {}, 'Low') 
Map.add_layer(med_fc.draw(color="FF8800", strokeWidth=5), {}, 'Medium') 
Map.add_layer(sev_fc.draw(color='FF0000', strokeWidth=5), {}, 'Severe') 


Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

# References
[1] ReefBase, “Coral Bleaching Data.” Harvard Dataverse, Feb. 06, 2025. doi: 10.7910/DVN/KUVQKY.